# 📋 DAT09 · 미션8 대량 입력과 저장 실패 시험하기 (학습자용)

---

## 실습 시간·작업 폴더·연결 파일

- **예상 실습 시간**: 3.5시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_practice`
- **연결 파일**: 첫 셀에서 `DAT09.md`를 연결하세요.

## 이전 미션 연계와 실행 결과

- **이전 미션 연계**: 같은 작업 폴더에서 미션1~미션7을 먼저 실행했다면, 첫 셀이 `src/m01.py`~`src/m07.py`를 불러와 작업 사전 `c`를 자동으로 복원합니다.

- 각 단계 함수(`m08_s1`~`m08_s5`)의 `TODO`를 직접 작성한 뒤 셀을 실행하세요.
- 실행 결과는 `outputs/M08/DAT09_M08_result.html`과 `step_01.csv`~`step_05.csv`에 저장됩니다.



In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'student'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

def hydrate_previous_missions(c, up_to=7):
    """같은 작업 폴더에 저장된 이전 미션(src/m01.py~)을 순차 실행해 작업 사전 c를 복원합니다."""
    import importlib.util
    src_dir = c['root'] / 'src'
    missing = [k for k in range(1, up_to + 1) if not (src_dir / f'm{k:02d}.py').is_file()]
    if missing:
        miss_str = ', '.join(f'미션{k}(DAT09_m{k:02d}_*.ipynb)' for k in missing)
        print(f'[안내] 아직 완료되지 않은 이전 미션이 있습니다: {miss_str}')
        print('이전 미션 노트북을 같은 폴더에서 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    for k in range(1, up_to + 1):
        if k in c['completed']:
            continue
        mod_path = src_dir / f'm{k:02d}.py'
        spec = importlib.util.spec_from_file_location(f'dat09_prev_m{k:02d}', mod_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fns = [getattr(mod, f'm{k:02d}_s{s}') for s in range(1, 6)]
        run_mission(c, k, fns)
    print(f'이전 미션(미션1~미션{up_to}) 상태 복원 완료.')
    return True

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
    hydrate_previous_missions(c, 7)
else:
    print('DAT09.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09.md'); hydrate_previous_missions(c)를 실행하세요.")


## 미션8 대량 입력과 저장 실패 시험하기

예상 3.5시간입니다.

#### 이번 미션의 목표

대량 입력을 실제 변환·검증·DB 저장까지 처리하고, 실패한 보고도 빠짐없이 기록하며 성능과 업무시간 가정을 구별합니다.

### 1단계 세 규모의 실제 부하 시험 입력 만들기

#### 이 단계에서 하는 일

작은 파일의 실행시간을 곱해서 대량 처리 성능이라고 할 수 없습니다. 실제 크기의 입력을 만들고 정상·오류·재전송 건수를 알고 시험해야 처리 누락을 찾을 수 있습니다.

#### 시작 전 확인

미션7까지 완료한 결과를 사용합니다. 노트북의 m08_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

generate_data.py의 create_load와 config/load_profile.json을 사용합니다. 여기서 원본 건수는 서로 다른 이벤트 수이며, 재전송은 같은 이벤트를 한 번 더 보내는 입력입니다.

#### 작업 순서

- 고유 이벤트 5,000·20,000·50,000건을 각각 생성하고 각 규모의 1%인 50·200·500건을 재전송으로 더하세요. 실제 파일의 줄 수를 세어 확인합니다.

- 고유 이벤트의 0.4%인 20·80·200건에는 수량 형식 오류를 넣으세요. 같은 오류 이벤트의 재전송이 있다면 오류 수신 건수와 고유 오류 수는 다를 수 있습니다.

- 생성 시드와 입력 조건, 파일 경로·해시·고유 건수·전체 수신 건수·고유 오류 수를 저장하세요. 생성한 파일을 다음 단계에서 실제로 읽습니다.


#### 저장할 결과

data/load_5000.jsonl, load_20000.jsonl, load_50000.jsonl과 각 manifest, outputs/M08/load_manifest.json을 저장합니다.

#### 직접 확인하는 방법

전체 수신은 5,050·20,200·50,500건입니다. 고유 오류는 20·80·200건이고 재전송과 구별해야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M08/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M08_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 기존 처리 함수를 큐와 DB 저장에 연결하기

#### 이 단계에서 하는 일

문자열을 읽는 속도만 재면 실제 업무 처리 성능을 알 수 없습니다. 앞서 만든 변환·검증·이벤트 처리를 거쳐 DB에 확정 저장하고, 각 수신의 상태를 남기는 범위까지 측정합니다.

#### 시작 전 확인

미션8의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m08_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

세 부하 입력, 미션2 normalize_fn·status_fn, 미션3 rule_fn, 미션4 이벤트 처리 기준과 SQLite를 사용합니다. 큐는 처리 전 보고가 기다리는 목록이며 이 실습은 Python 스레드로 유입을 모의합니다.

#### 작업 순서

- A·B·C 출처별 생산자 스레드 세 개가 이벤트를 큐에 넣고 소비자 한 개가 꺼내게 하세요. 생산자 종료 신호를 모두 받은 뒤 남은 보고까지 처리합니다.

- 500건씩 묶어 변환·검증·중복 ID·버전 처리를 수행하세요. DB에는 이미 본 ID, 대상별 현재 상태, 수신별 상태 원장을 저장합니다. 500건 미만인 마지막 묶음도 빠뜨리지 않습니다.

- 각 실행마다 새 시험 DB를 만들고 한 묶음의 저장을 트랜잭션으로 처리하세요. 성공한 commit 뒤에만 저장 완료로 기록합니다.

- 파일 읽기부터 변환·검증·DB 저장·상태 기록까지 perf_counter로 시간을 재세요. 독립 검산 시간은 별도이며 측정 범위에 넣었는지 명시합니다. 5천 규모로 워밍업 한 번을 실행합니다.


#### 저장할 결과

실제 처리 함수 c['process_load_fn'], 워밍업 결과, 실행별 SQLite DB·event_status.csv·queue_log.json·retry_log.json을 남깁니다.

#### 직접 확인하는 방법

큐가 비어 보인다는 이유만으로 생산자가 끝나기 전에 종료하면 안 됩니다. 워밍업도 입력 수와 상태 원장 합계가 같고 대상별 검산 차이가 0인지 확인합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M08/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M08_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 저장 실패 후 재시도와 최종 실패 기록 확인하기

#### 이 단계에서 하는 일

저장이 실패했는데 메모리에서만 처리 완료로 남으면 재시도 때 보고가 누락될 수 있습니다. 반대로 일부 저장을 남기면 재시도 때 중복될 수 있습니다. 실패한 묶음을 되돌린 뒤 다시 시도하는지 확인합니다.

#### 시작 전 확인

미션8의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m08_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

5,050건 입력과 process_load_fn을 사용합니다. 정상 실행과 별도의 시험 DB에서 네 번째 묶음의 commit 직전에 오류를 발생시키세요.

#### 작업 순서

- 첫 시도만 실패하는 시험을 만드세요. 트랜잭션을 rollback한 뒤 같은 묶음을 재시도해 한 번만 저장되는지 확인합니다.

- 같은 묶음이 세 번 모두 실패하는 시험도 만드세요. 세 번 뒤에는 그 수신을 FAILED로 원장에 기록하고, 오류 없이 저장됐다고 세지 않습니다.

- 실패 묶음·시도 번호·오류·rollback 여부·최종 상태를 남기세요. 일시 실패에서는 독립 일괄 검산과 대상별 결과가 같아야 합니다.

- 영구 실패에서는 정상 결과와 차이가 날 수 있음을 보고하되, 그 차이를 감추지 말고 FAILED 수와 상태 합계를 확인하세요. 수신된 모든 보고에 최종 상태가 있어야 합니다.


#### 저장할 결과

failure_tests.json과 두 시험의 retry_log·queue_log·event_status·DB를 남깁니다.

#### 직접 확인하는 방법

일시 실패는 FAILED=0, 대상별 차이=0이어야 합니다. 영구 실패는 FAILED가 0보다 크고 전체 상태 합계가 5,050건이어야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M08/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M08_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 규모별 세 번 실행하고 수신 건수 대조하기

#### 이 단계에서 하는 일

성능은 실행마다 달라지므로 한 번의 가장 빠른 값만 고르면 결과가 왜곡됩니다. 정확성도 최종 합계만 보지 않고 각 대상의 버전과 수량을 별도 계산과 맞춰야 합니다.

#### 시작 전 확인

미션8의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m08_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

워밍업을 마친 처리 함수, 세 입력과 독립 일괄 검산을 사용합니다. 각 회차는 새 DB에서 시작하고 같은 입력 해시를 유지합니다.

#### 작업 순서

- 5천·2만·5만 원본 규모마다 세 번씩, 총 아홉 번 실행하세요. 각 회차에 일시 commit 실패와 재시도를 포함해 복구 후 결과를 측정합니다.

- 수신 수·고유 ID 수·고유 오류 수와 APPLIED·DUPLICATE·STALE·QUARANTINED·CONFLICT·FAILED 등 실제 나온 상태를 기록하세요. 각 수신은 상태 하나에만 속하게 합니다.

- 정상 고유 이벤트 처리율은 정상 고유 이벤트 중 적용 또는 과거 버전으로 처리 완료한 수/정상 고유 이벤트 수로 계산하세요. 오류 격리를 정상 재고 반영에 더하지 않습니다.

- 전체 이벤트를 별도로 묶어 최대 유효 버전을 구한 결과와 DB의 대상별 버전·수량을 비교하세요. 환경·Python·pandas·묶음 크기·동시 처리 방식도 저장합니다.


#### 저장할 결과

load_results.csv에 아홉 회차를, runtime_environment.json에 환경을 저장하고 회차별 상태·재시도·검산 근거를 남깁니다.

#### 직접 확인하는 방법

아홉 번 모두 수신=상태합, 대상별 차이 0개, 정상 고유 이벤트 처리율 100%인지 확인하세요. 상태별 건수는 스레드 수신 순서에 따라 달라질 수 있습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M08/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M08_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 실측 성능과 업무시간 절감 가정 나누어 보고하기

#### 이 단계에서 하는 일

프로그램의 초 단위 처리시간과 사람이 검토·승인을 기다리는 분 단위 업무시간은 다릅니다. 둘을 섞으면 실제 확인하지 않은 업무 개선을 실측 성과처럼 보고하게 됩니다.

#### 시작 전 확인

미션8의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m08_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

load_results.csv의 아홉 회차와 workflow의 실습 가정을 사용합니다. 실제 처리시간은 규모별 세 회차의 중앙값·최대, 이벤트 지연은 큐에 넣은 때부터 저장 완료까지의 p95로 보고합니다.

#### 작업 순서

- 규모별 실제 처리시간 중앙값·최대·이벤트 지연 p95와 자동 처리 상태 분류율을 정리하세요. 자동 분류율은 적용·중복·과거·격리로 끝난 수신/전체 수신이며 정상 재고 반영률과 다릅니다.

- 업무시간은 수작업 1수신당 0.4분, 자동화 후 검토 1수신당 0.01분, 고유 오류 확인 1건당 4분, 승인 대기 30분으로 가정하세요. 자동화 후 시간에는 실제 처리시간 중앙값을 분으로 바꿔 더합니다.

- 절감률=(수작업 가정시간-자동화 후 가정시간)/수작업 가정시간을 계산하고 승인 대기를 60분으로 바꾼 값도 비교하세요.

- 5만 원본의 300초, 자동 처리 상태 분류율 99%, 가정 절감률 80% 목표 충족 여부를 별도로 적으세요. 목표에 못 미친 값도 그대로 남깁니다.


#### 저장할 결과

load_report.md와 규모별 요약 표를 저장하고 실측·가정·목표를 구분합니다.

#### 직접 확인하는 방법

초를 분으로 바꿨는지, 승인 대기 30분이 빠지지 않았는지 확인하세요. 자동 격리가 많아도 재고가 정확해졌다는 뜻은 아닙니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M08/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M08_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 세 규모의 실제 부하 시험 입력 만들기
# 할 일: 고유 이벤트 5,000·20,000·50,000건을 각각 생성하고 각 규모의 1%인 50·200·500건을 재전송으로 더하세요. 실제 파일의 줄 수를 세어 확인합니다. 고유 이벤트의 0.4%인 20·80·200건에는 수량 형식 오류를 넣으세요. 같은 오류 이벤트의 재전송이 있다면 오류 수신 건수와 고유 오류 수는 다를 수 있습니다. 생성 시드와 입력 조건, 파일 경로·해시·고유 건수·전체 수신 건수·고유 오류 수를 저장하세요. 생성한 파일을 다음 단계에서 실제로 읽습니다.
# 확인: 전체 수신은 5,050·20,200·50,500건입니다. 고유 오류는 20·80·200건이고 재전송과 구별해야 합니다.
def m08_s1(c):
    """세 규모의 실제 부하 시험 입력 만들기.

    읽을 작업값: root
    저장할 작업값: load_manifest
    결과 폴더: folder(c, 8)
    """
    return TODO(8, 1)

# STEP 02 기존 처리 함수를 큐와 DB 저장에 연결하기
# 할 일: A·B·C 출처별 생산자 스레드 세 개가 이벤트를 큐에 넣고 소비자 한 개가 꺼내게 하세요. 생산자 종료 신호를 모두 받은 뒤 남은 보고까지 처리합니다. 500건씩 묶어 변환·검증·중복 ID·버전 처리를 수행하세요. DB에는 이미 본 ID, 대상별 현재 상태, 수신별 상태 원장을 저장합니다. 500건 미만인 마지막 묶음도 빠뜨리지 않습니다. 각 실행마다 새 시험 DB를 만들고 한 묶음의 저장을 트랜잭션으로 처리하세요. 성공한 commit 뒤에만 저장 완료로 기록합니다. 파일 읽기부터 변환·검증·DB 저장·상태 기록까지 perf_counter로 시간을 재세요. 독립 검산 시간은 별도이며 측정 범위에 넣었는지 명시합니다. 5천 규모로 워밍업 한 번을 실행합니다.
# 확인: 큐가 비어 보인다는 이유만으로 생산자가 끝나기 전에 종료하면 안 됩니다. 워밍업도 입력 수와 상태 원장 합계가 같고 대상별 검산 차이가 0인지 확인합니다.
def m08_s2(c):
    """기존 처리 함수를 큐와 DB 저장에 연결하기.

    읽을 작업값: SyncEngine, catalog, normalize_fn, root, rule_fn, status_fn
    저장할 작업값: process_load_fn, warmup
    결과 폴더: folder(c, 8)
    """
    return TODO(8, 2)

# STEP 03 저장 실패 후 재시도와 최종 실패 기록 확인하기
# 할 일: 첫 시도만 실패하는 시험을 만드세요. 트랜잭션을 rollback한 뒤 같은 묶음을 재시도해 한 번만 저장되는지 확인합니다. 같은 묶음이 세 번 모두 실패하는 시험도 만드세요. 세 번 뒤에는 그 수신을 FAILED로 원장에 기록하고, 오류 없이 저장됐다고 세지 않습니다. 실패 묶음·시도 번호·오류·rollback 여부·최종 상태를 남기세요. 일시 실패에서는 독립 일괄 검산과 대상별 결과가 같아야 합니다. 영구 실패에서는 정상 결과와 차이가 날 수 있음을 보고하되, 그 차이를 감추지 말고 FAILED 수와 상태 합계를 확인하세요. 수신된 모든 보고에 최종 상태가 있어야 합니다.
# 확인: 일시 실패는 FAILED=0, 대상별 차이=0이어야 합니다. 영구 실패는 FAILED가 0보다 크고 전체 상태 합계가 5,050건이어야 합니다.
def m08_s3(c):
    """저장 실패 후 재시도와 최종 실패 기록 확인하기.

    읽을 작업값: process_load_fn, root
    저장할 작업값: load_failure_tests
    결과 폴더: folder(c, 8)
    """
    return TODO(8, 3)

# STEP 04 규모별 세 번 실행하고 수신 건수 대조하기
# 할 일: 5천·2만·5만 원본 규모마다 세 번씩, 총 아홉 번 실행하세요. 각 회차에 일시 commit 실패와 재시도를 포함해 복구 후 결과를 측정합니다. 수신 수·고유 ID 수·고유 오류 수와 APPLIED·DUPLICATE·STALE·QUARANTINED·CONFLICT·FAILED 등 실제 나온 상태를 기록하세요. 각 수신은 상태 하나에만 속하게 합니다. 정상 고유 이벤트 처리율은 정상 고유 이벤트 중 적용 또는 과거 버전으로 처리 완료한 수/정상 고유 이벤트 수로 계산하세요. 오류 격리를 정상 재고 반영에 더하지 않습니다. 전체 이벤트를 별도로 묶어 최대 유효 버전을 구한 결과와 DB의 대상별 버전·수량을 비교하세요. 환경·Python·pandas·묶음 크기·동시 처리 방식도 저장합니다.
# 확인: 아홉 번 모두 수신=상태합, 대상별 차이 0개, 정상 고유 이벤트 처리율 100%인지 확인하세요. 상태별 건수는 스레드 수신 순서에 따라 달라질 수 있습니다.
def m08_s4(c):
    """규모별 세 번 실행하고 수신 건수 대조하기.

    읽을 작업값: checks, load_failure_tests, process_load_fn, root
    저장할 작업값: load_results
    결과 폴더: folder(c, 8)
    """
    return TODO(8, 4)

# STEP 05 실측 성능과 업무시간 절감 가정 나누어 보고하기
# 할 일: 규모별 실제 처리시간 중앙값·최대·이벤트 지연 p95와 자동 처리 상태 분류율을 정리하세요. 자동 분류율은 적용·중복·과거·격리로 끝난 수신/전체 수신이며 정상 재고 반영률과 다릅니다. 업무시간은 수작업 1수신당 0.4분, 자동화 후 검토 1수신당 0.01분, 고유 오류 확인 1건당 4분, 승인 대기 30분으로 가정하세요. 자동화 후 시간에는 실제 처리시간 중앙값을 분으로 바꿔 더합니다. 절감률=(수작업 가정시간-자동화 후 가정시간)/수작업 가정시간을 계산하고 승인 대기를 60분으로 바꾼 값도 비교하세요. 5만 원본의 300초, 자동 처리 상태 분류율 99%, 가정 절감률 80% 목표 충족 여부를 별도로 적으세요. 목표에 못 미친 값도 그대로 남깁니다.
# 확인: 초를 분으로 바꿨는지, 승인 대기 30분이 빠지지 않았는지 확인하세요. 자동 격리가 많아도 재고가 정확해졌다는 뜻은 아닙니다.
def m08_s5(c):
    """실측 성능과 업무시간 절감 가정 나누어 보고하기.

    읽을 작업값: load_results
    저장할 작업값: load_summary
    결과 폴더: folder(c, 8)
    """
    return TODO(8, 5)

result_html = run_mission(c, 8, [m08_s1, m08_s2, m08_s3, m08_s4, m08_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
